## Lab 8- Introduction to Machine Learning
Based on Chapter 2 from Geron's book, Hands-on Machine Learning with Scikit-Learn Keras & Tensorflow

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/dtrad/geoml_course/blob/master/Practice8-IntroToML.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os
#print('directory',os.environ['PWD'])
# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

## Loading and examining data

In [ ]:
import os
import tarfile
import urllib.request

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

In [ ]:
fetch_housing_data()

In [ ]:
pwd

In [ ]:
ls datasets/housing

In [ ]:
import pandas as pd

def load_housing_data(housing_path=HOUSING_PATH):    
    csv_path = os.path.join(housing_path, "housing.csv")
    print("reading %s"%csv_path)
    return pd.read_csv(csv_path)

In [ ]:
housing=load_housing_data()
print("created a panda object",type(housing))
housing.head()

In [ ]:
housing.describe()

In [ ]:
housing.info()

In [ ]:
housing.ocean_proximity

In [ ]:
housing.ocean_proximity.value_counts()

In [ ]:
housing.hist(bins=20,figsize=(10,10),edgecolor='b')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
housing["median_income"].hist(edgecolor='b',bins=20)

In [ ]:
housing.describe()

In [ ]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

In [ ]:
(housing.keys())

In [ ]:
housing.income_cat.hist()

Let us use a Stratified ShuffleSplit cross-validator that provides train/test indices to split data in train/test sets.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

In [ ]:
#split?

In [ ]:
type(strat_train_set),type(split),type(housing)

In [ ]:
strat_train_set.info()

Categoricals are a pandas data type corresponding to categorical variables in statistics. \
A categorical variable takes on a limited, and usually fixed, number of possible values\

In [ ]:
strat_train_set.income_cat.value_counts()/len(strat_train_set)

In [ ]:
strat_test_set.income_cat.value_counts()/len(strat_test_set)

Now we can some visualization and also some data preparation. We start with a fresh copy form strat_train_set.

In [ ]:
strat_train_set.head(2)

In [ ]:
housing=strat_train_set.copy()

In [ ]:
housing.plot(kind='scatter',x='latitude',y='longitude',c='median_house_value',cmap='jet',s=housing.population/100,figsize=(10,8))

In [ ]:
corr_matrix=housing.corr()

In [ ]:
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix
attributes = ["median_house_value", "median_income", "total_rooms",
              "housing_median_age"]
scatter_matrix(housing[attributes],figsize=(10,10))

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]

In [ ]:
housing.head()

In [ ]:
print(len(housing.keys()))
print(housing.keys())

# Preparing data
We will add these attributes later again but setting everything as a pipeline dataflow.\
But first let us check for features that could cause problems for ML algorithms.\
There are two problems: a) missing values b) features that do not map to numbers.\
Also, we need to separate from the data the features we want to use for labels and leave only the features used for calculation

Let us remove the label from the data and create a "series" (dataframe with one feature) to use in fitting.\
We will do this in a fresh data set directly from the train_set generator.

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1) # drop labels for training set
housing_labels = strat_train_set["median_house_value"].copy()

In [ ]:
print(type(strat_train_set),strat_train_set.shape)
print(type(housing),housing.shape)
print(type(housing_labels),housing_labels.shape)

In [ ]:
housing_labels[:5]

In [ ]:
housing.head(5)

In [ ]:
print(len(housing.keys()))
print(housing.keys())

Need to check for missing entries and remove them from the data or fill them up before ML algorithms

In [ ]:
sample_incomplete_rows = housing[housing.isnull().any(axis=1)]
sample_incomplete_rows

In [ ]:
sample_incomplete_rows.info()

In this case we will fill them up with the median of all the other variables using the class Imputer

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")

In [ ]:
#imputer?

In [ ]:
#but need to take out the string keys where the median does not work
housing_num = housing.drop("ocean_proximity", axis=1)

In [ ]:
X=imputer.fit_transform(housing_num)

In [ ]:
type(X),X.shape

The output from inputer is a ndarray, so we need to rebuild a panda dataframe

In [ ]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns,index=housing_num.index)

This data frame is the same as before except that the incomplete rows are filled in by the imputer.

In [ ]:
housing_tr.loc[sample_incomplete_rows.index.values]

Let us check again, we got not left null values

In [ ]:
housing_tr[housing_tr.isnull().any(axis=1)]

In [ ]:
housing_tr.info()

In [ ]:
housing.info()

All these operations we did are going to be difficult to remember every time we have more data.\
sklearn provides pipelines, that are structures of processing steps.\
These processing steps have to be encapsulated in a method or a class.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# column index
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self  # nothing else to do
    def transform(self, X):
        #print("class CombinedAttributesAdder: input is a ",type(X))
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values)

In [ ]:
# Reminder, concatenating np.arrays with np.c_ 
aa=np.array([1,2])
bb=np.array([3,4])
aa=aa[:,np.newaxis]
bb=bb[:,np.newaxis]
cc=np.c_[aa,bb]
print(aa.shape,bb.shape,cc.shape)
print(cc)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('attribs_adder', CombinedAttributesAdder()),
        ('std_scaler', StandardScaler()),
    ])

housing_tr = num_pipeline.fit_transform(housing_num)

In [ ]:
# housing num the input is a dataframe
print(type(housing_num),housing_num.shape)
# but housing_tr is a ndarray (not dataframe)
print(type(housing_tr),housing_tr.shape)

Think of pipelines as a line of processes connected through pipes.\
Pipes are the typical communication method between programs in linux.\
Also, in R we saw pipes. The functions we saw in R using pipes are the equivalent of these pipelines.\
Pipelines are common in other environments, for example seismic processing packages.


## Let us run regression with several machine learning algorithms
We will start from the housing_num frame where we already dropped the non-numerical column ocean proximity.

In [ ]:
housing_num.head()

In [ ]:
housing_labels[:5]

### Linear regression and decision tree
For each method we will start from housing with numbers only, run the pipeline that interpolates, adds attributes and scales.\
Then we just apply the linear regression algorithm from sklearn by using fit. 

In [ ]:
# Example for a linear regresion in sklearn.
from sklearn.linear_model import LinearRegression

#create data
x = 2*np.random.rand(100,1)
y=4+3*x+np.random.randn(100,1)
lin_reg=LinearRegression()
lin_reg.fit(x,y)
lin_reg.intercept_,lin_reg.coef_
xn=x;
lin_reg.predict(xn)
yn=lin_reg.predict(xn)
plt.plot(x,y,'.',xn,yn)
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
housing_prepared=num_pipeline.fit_transform(housing_num)
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

In [ ]:
print("housing_num",housing_num.shape,type(housing_num))
print("housing_prepared",housing_prepared.shape, housing_prepared.dtype)
print("housing_labels",housing_labels.shape, housing_labels.dtype)

In [ ]:
housing_labels.hist(bins=50,edgecolor='b')

In [ ]:
# let's try the full preprocessing pipeline
ninstances=housing_num.shape[0]
print(ninstances)
some_data = housing_num[:ninstances]
some_labels = housing_labels.iloc[:ninstances]
some_data_prepared = num_pipeline.transform(some_data)
predictions=lin_reg.predict(some_data_prepared)
print("labels - predictions\n"),[print('{:.0f}'.format(some_labels.iloc[i]),'{:.0f}'.format(predictions[i])) for i in range(10)]
print()

In [ ]:
print(type(some_data),type(some_labels),type(some_data_prepared),type(predictions))

In [ ]:
fig, ax = plt.subplots(1)
ax.hist(predictions,bins=150,edgecolor='b')
ax.set_xlim(0,500000)

In [ ]:
housing.head()

In [ ]:
print(some_labels.iloc[0])
print(some_labels.loc[0])
print(some_labels.head(1))

Let us evaluate how good the Mean Squared Error (MSE) is when all data are considered

In [ ]:
housing_labels_np=housing_labels.to_numpy(copy=True)
housing_labels_np[0]

In [ ]:
from sklearn.metrics import mean_squared_error
housing_prepared=num_pipeline.transform(housing_num)
housing_predictions = lin_reg.predict(housing_prepared)
lin_mse = mean_squared_error(housing_labels_np, housing_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

The score for the linear regressor is not great. Let us try a decision tree:

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)

In [ ]:
housing_predictions = tree_reg.predict(housing_prepared)
tree_mse = mean_squared_error(housing_labels, housing_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

An error of zero it probably means overfitting. We need to use a more objective measure than just the data
where the regressor was calculated. We could use the validation and test datasets.
Another more automatic way of doing this, is using cross-validation. 
The data are separated on a training and test data sets many times, and each time the
algorithm is trained again. 

### Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score
tree_scores = cross_val_score(tree_reg, housing_prepared, housing_labels,scoring="neg_mean_squared_error", cv=10)

In [ ]:
# to visualize the scores we can use a panda series like this
pd.Series(np.sqrt(-tree_scores)).describe()

In [ ]:
# or we can create a customized function
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())
    
tree_rmse_scores = np.sqrt(-tree_scores)
display_scores(tree_rmse_scores)

In [ ]:
#let us repeat for the linear regressor
lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error",cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)

In [ ]:
display_scores(lin_rmse_scores)

We see that the linear regressor in reality did better than the regression tree.
A regression tree alone is not that powerful to generalize because it tends to overfit.
We will see that in future lectures. 
However, a more powerful device that uses trees is a Random forest. 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=10)
forest_reg.fit(housing_prepared, housing_labels)

In [ ]:
housing_predictions = forest_reg.predict(housing_prepared)
forest_mse = mean_squared_error(housing_labels, housing_predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

In [ ]:
forest_scores = cross_val_score(forest_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error",cv=10)

In [ ]:
forest_rmse_scores = np.sqrt(-forest_scores)
display_scores(forest_rmse_scores)

In [ ]:
from sklearn.svm import SVR
svm_reg = SVR(kernel="linear")
svm_reg.fit(housing_prepared, housing_labels)
housing_predictions = svm_reg.predict(housing_prepared)
svm_mse = mean_squared_error(housing_labels, housing_predictions)
svm_rmse = np.sqrt(svm_mse)
svm_rmse

### GRID SEARCH
This process could take a long time because each method has many options to optimize.
To automatize this tuning process skit-learn has a function that can systematically try ranges of parameters
combined with cross-validations


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    # try 12 (3×4) combinations of hyperparameters
    {'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
    # then try 6 (2×3) combinations with bootstrap set as False
    {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
  ]

forest_reg = RandomForestRegressor(random_state=42, n_jobs=10)
# train across 5 folds, that's a total of (12+6)*5=90 rounds of training 
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
grid_search.fit(housing_prepared, housing_labels)

In [ ]:
# Best parameters
grid_search.best_params_

In [ ]:
# Best estimator
grid_search.best_estimator_

In [ ]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print('{:.0f}'.format(np.sqrt(-mean_score)), params)

there are ways to search the whole model space that could improve this result but for this test
let us settle with this result:

In [ ]:
final_model = grid_search.best_estimator_

Now we will finish the predictions with this estimator.
We will take the strat_test_set and apply the preprocessing:

In [ ]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()
#but need to take out the string keys where the median does not work
X_test_num = X_test.drop("ocean_proximity", axis=1)
X_test_prepared = num_pipeline.transform(X_test_num)

In [ ]:
final_predictions = final_model.predict(X_test_prepared)

final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)

In [ ]:
print(final_rmse)

In [ ]:
print('predictions, labels')
[print('{:.0f}\t{:.0f}'.format(final_predictions[i], y_test.iloc[i])) for i in range(10)]
print()